# Bayesian Personalized Ranking Matrix Factorization (BPR-MF)

In [1]:
import gdown, zipfile, os

# Skip if already downloaded
if not os.path.exists('dataset/processed'):
    FILE_ID = '1a7kCwqP2Vhlv43eep0ODdzGVR6v2ouP6'
    gdown.download(id=FILE_ID, output='processed.zip', quiet=False)
    with zipfile.ZipFile('processed.zip', 'r') as z:
        z.extractall('dataset/')
    os.remove('processed.zip')
    print('Downloaded.')
else:
    print('Already exists, skipping download.')

Downloading...
From (original): https://drive.google.com/uc?id=1a7kCwqP2Vhlv43eep0ODdzGVR6v2ouP6
From (redirected): https://drive.google.com/uc?id=1a7kCwqP2Vhlv43eep0ODdzGVR6v2ouP6&confirm=t&uuid=c6a6c4ed-14d2-4072-901c-67253d389ef5
To: /content/processed.zip
100%|██████████| 540M/540M [00:07<00:00, 73.1MB/s]


Downloaded.


In [2]:
# Install implicit
!pip install implicit -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 2.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [5]:
# Libraries
import numpy as np
import pandas as pd
import scipy.sparse as sp
from tqdm.auto import tqdm
import pickle
import random
from implicit.bpr import BayesianPersonalizedRanking

train_df = pd.read_csv('dataset/processed/train.csv')
val_df   = pd.read_csv('dataset/processed/val.csv')
test_df  = pd.read_csv('dataset/processed/test.csv')
games_df = pd.read_csv('dataset/processed/games_cleaned.csv')
ratings_df = pd.read_csv('dataset/processed/user_ratings_cleaned.csv')

with open('dataset/processed/id_maps.pkl', 'rb') as f:
    maps = pickle.load(f)

n_users = len(maps['user2idx'])
n_items = len(maps['item2idx'])
print(f'n_users: {n_users:,} | n_items: {n_items:,}')
print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')

n_users: 272,184 | n_items: 21,922
Train: 18,151,978 | Val: 272,184 | Test: 272,184


In [6]:
# BPR focuses on whether an interaction happened, not the specific rating value
interactions = ratings_df[["Username", "BGGId"]].drop_duplicates()

# Create mappings
all_users = interactions["Username"].unique()
all_items = interactions["BGGId"].unique()
user2idx = {u: i for i, u in enumerate(all_users)}
item2idx = {g: i for i, g in enumerate(all_items)}
idx2user = {i: u for u, i in user2idx.items()}
idx2item = {i: g for g, i in item2idx.items()}
n_users = len(all_users)
n_items = len(all_items)
interactions["user_idx"] = interactions["Username"].map(user2idx)
interactions["item_idx"] = interactions["BGGId"].map(item2idx)

user_positives = interactions.groupby("user_idx")["item_idx"].apply(set).to_dict()
print(f"Dataset stats: \n{n_users:,} users \n{n_items:,} items \n{len(interactions):,} interactions")

Dataset stats: 
272,184 users 
21,922 items 
18,663,786 interactions


In [7]:
train_positives = {}
test_pairs = []

for user_idx, pos_set in user_positives.items():
    pos_list = list(pos_set)
    if len(pos_list) < 2:
        train_positives[user_idx] = pos_set
        continue
    held_out = random.choice(pos_list)
    test_pairs.append((user_idx, held_out))
    train_positives[user_idx] = pos_set - {held_out}
print(f"Training on {sum(len(v) for v in train_positives.values()):,} items")

Training on 18,391,607 items


# BPR-MF Model
We use `implicit.bpr.BayesianPersonalizedRanking` which implements the same BPR-MF algorithm.

The `fit()` method expects a **user × item** CSR sparse matrix of implicit feedback (1 = interaction).

In [8]:
# Build the user-item CSR matrix from train_positives
rows, cols = [], []
for u_idx, pos_set in train_positives.items():
    for i_idx in pos_set:
        rows.append(u_idx)
        cols.append(i_idx)

train_matrix = sp.csr_matrix(
    (np.ones(len(rows), dtype=np.float32), (rows, cols)),
    shape=(n_users, n_items)
)

print(f"Train matrix shape: {train_matrix.shape}  |  nnz: {train_matrix.nnz:,}")

Train matrix shape: (272184, 21922)  |  nnz: 18,391,607


In [9]:
# Hyperparameters
N_FACTORS  = 32
LR         = 0.01
REG        = 0.001
N_EPOCHS   = 10

model = BayesianPersonalizedRanking(
    factors=N_FACTORS,
    learning_rate=LR,
    regularization=REG,
    iterations=N_EPOCHS,
    random_state=42,
    verify_negative_samples=True,   # mirrors the original's rejection-sampling loop
)

# fit() expects a user x item CSR matrix
model.fit(train_matrix)

  0%|          | 0/10 [00:00<?, ?it/s]

# Evaluation
Metric: Hit Rate @ K
For each test (user, held-out item) pair, we ask: Is the held-out item in the model's top-K recommendations?

Hit Rate = (number of hits) / (number of test users)

In [26]:
def evaluate(score_series, train_df, test_df, n_negatives=99, K=10, seed=42):
    """
    score_series: pd.Series indexed by item_idx with popularity score
    For each test user: rank 1 positive + n_negatives random unseen items
    """
    rng = np.random.default_rng(seed)
    all_items = np.array(list(maps['item2idx'].values()))

    # Items seen per user in training
    seen = train_df.groupby('user_idx')['item_idx'].apply(set).to_dict()

    hits, ndcgs, mrrs = [], [], []

    for _, row in test_df.iterrows():
        uid      = int(row['user_idx'])
        pos_item = int(row['item_idx'])

        # Sample negatives - items user hasn't seen
        user_seen  = seen.get(uid, set()) | {pos_item}
        candidates = np.setdiff1d(all_items, list(user_seen))
        neg_items  = rng.choice(candidates, size=n_negatives, replace=False)

        # Score and rank all candidates
        items_to_rank = np.append(neg_items, pos_item)
        scores = score_series.reindex(items_to_rank).fillna(0).values
        ranked = items_to_rank[np.argsort(-scores)]

        # Find position of positive item
        pos_rank = np.where(ranked == pos_item)[0][0] + 1  # 1-indexed

        hits.append(1 if pos_rank <= K else 0)
        ndcgs.append(np.log(2) / np.log(pos_rank + 1) if pos_rank <= K else 0)
        mrrs.append(1 / pos_rank if pos_rank <= K else 0)

    return {
        f'HR@{K}':   round(np.mean(hits), 4),
        f'NDCG@{K}': round(np.mean(ndcgs), 4),
        f'MRR@{K}':  round(np.mean(mrrs), 4),
    }

# Results

In [36]:
# Build score_series from BPR-MF model scores
# Get item scores averaged across all users (or use item factors norm)
item_scores = np.linalg.norm(model.item_factors, axis=1)
score_series = pd.Series(item_scores, index=np.arange(n_items))

# Map train_df and test_df to use user_idx / item_idx columns
train_eval = interactions.rename(columns={'user_idx': 'user_idx', 'item_idx': 'item_idx'})
test_eval  = pd.DataFrame(test_pairs, columns=['user_idx', 'item_idx'])
results = evaluate(score_series, train_eval, test_eval)
results_df = pd.DataFrame([results], index=['BPR-MF'])
print(results_df)

         HR@10  NDCG@10  MRR@10
BPR-MF  0.7861    0.547  0.4723


# Sample Recommendations

In [35]:
# Add game names and ratings for readability
idx_to_bggid = idx2item
games_lookup = games_df.set_index("BGGId")[["Name", "AvgRating"]]

# Global ranking by average rating
global_ranking = games_df.set_index("BGGId")["AvgRating"].sort_values(ascending=False)

def get_recommendations(user_idx, train_df, top_k=10):
    seen = set(train_df[train_df["user_idx"] == user_idx]["item_idx"])
    # Get BPR-MF scores for all items, exclude seen
    top_idxs, top_scores = model.recommend(
        user_idx,
        train_matrix[user_idx],
        N=top_k,
        filter_already_liked_items=True
    )
    bgg_ids = [idx_to_bggid[i] for i in top_idxs]
    recs = games_lookup.loc[bgg_ids].reset_index()
    recs["BPR_Score"] = top_scores
    return recs

# Sample 3 users
uid = random.choice(list(user2idx.values()))
username = idx2user[uid]
print(f"Top 10 recommendations for user {username}:")
print(get_recommendations(uid, interactions).to_string(index=False))

Top 10 recommendations for user summonschy:
 BGGId              Name  AvgRating  BPR_Score
160495           ZhanGuo    7.57869   4.347291
 70149     Ora et Labora    7.70347   4.300577
162082              Deus    7.31325   4.278121
 43015   Hansa Teutonica    7.68083   3.924469
 72225               CO₂    7.14314   3.909797
 66505 The Speicherstadt    7.02879   3.594850
164338   The Golden Ages    7.27714   3.555285
 62219  Dominant Species    7.83930   3.493498
154246           La Isla    6.87517   3.492789
169794      Haspelknecht    7.25245   3.403358
